## Tutorial for standalone prediction with OCTRON
Use `predict_batch()` without the GUI for batch video prediction.

In [1]:
import sys
from loguru import logger
logger.remove()  
logger.add(sys.stderr, level="ERROR")  # only show errors

1

In [6]:
# Basic imports for octron
import octron
from octron.yolo_octron.yolo_octron import YOLO_octron
from octron.yolo_octron.constants import DEFAULT_REGION_PROPERTIES

from pathlib import Path
octron_base_path = Path(octron.__file__).resolve().parent

In [3]:
# This here is just to make your life easier 
# It finds the installed trackers and lists their names 
# so you can refer to them below (tracker_name = )

from octron.tracking.helpers.tracker_checks import (
    load_boxmot_trackers, 
    list_available_trackers
)

# List available trackers
trackers_yaml = octron_base_path / 'tracking/boxmot_trackers.yaml'
trackers_dict = load_boxmot_trackers(trackers_yaml)
available = list_available_trackers(trackers_dict)

for tracker_id, display_name in available.items():
    print(f'{tracker_id:15s}  ({display_name})')

ByteTrack        (ByteTrack)
OcSort           (OcSort)
BotSort          (BotSort)
D-OcSort         (D-OcSort)
HybridSort       (HybridSort)
BoostTrack       (BoostTrack)


In [4]:
# Initialize YOLO_octron
yolo = YOLO_octron(
    models_yaml_path=None,   # Optional: path to YOLO models YAML
    project_path=None,       # Optional: path to OCTRON project directory
    clean_training_dir=False
)

In [5]:
# Configure paths
video_paths = [
    Path('path/to/video.mp4'),
    Path('path/to/video.mp4'),
]
model_path = Path('path/to/your/custom-trained/model/best.pt')
tracker_name = 'hybridsort'  # case-insensitive; or 'BotSort', 'ocsort', 'D-OcSort', etc.
output_dir = None # None will save alongside original video, but you can change that to a folder on disk

In [ ]:
# Run batch prediction

for progress in yolo.predict_batch(
    videos=video_paths, # videos: single path (str/Path), list of paths, or pre-built dict (like in the GUI)
    model_path=model_path,
    device='mps',                # or 'cuda', 'cpu'
    tracker_name=tracker_name,   # case-insensitive name lookup
    # tracker_cfg_path=None,     # alternative: path to custom tracker YAML (overrides tracker_name)
    # tracker_params=None,       # dict of parameter overrides, e.g. {'det_thresh': 0.5}
    skip_frames=0,
    one_object_per_label=False,  # True = keep only highest-confidence detection per label
    region_properties=list(DEFAULT_REGION_PROPERTIES),  # None for bbox-only
    # extra_properties=None,     # tuple of custom callables for regionprops_table
    iou_thresh=0.7,
    conf_thresh=0.5,
    opening_radius=0,            # morphological opening radius for mask cleanup
    overwrite=True,
    buffer_size=500,
    output_dir=output_dir,       # custom output root; default (None): alongside each video file
):
    if progress['stage'] == 'processing':
        if progress['frame'] % 100 == 0:  # print every 100 frames
            print(
                f"Video {progress['video_index']+1}/{progress['total_videos']} "
                f"({progress['video_name']}): "
                f"Frame {progress['frame']}/{progress['total_frames']}"
            )
    elif progress['stage'] == 'video_complete':
        print(f"\n✓ Completed {progress['video_name']}")
        print(f"  Results saved to: {progress['save_dir']}\n")
    elif progress['stage'] == 'skipped_video':
        print(f"⏭ Skipped {progress['video_name']} (already exists)")
    elif progress['stage'] == 'complete':
        print(f"\n✓ All videos processed!")

## Custom measurement functions
Use `extra_properties` to pass callables to `regionprops_table`. Each function's `__name__` becomes a CSV column.

In [ ]:
import numpy as np

In [ ]:
# Mask-only function
def compactness(regionmask):
    area = np.sum(regionmask)
    perimeter = np.sum(regionmask ^ np.roll(regionmask, 1, axis=0))
    return (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

# Intensity-based function (receives the original video frame)
def mean_brightness(regionmask, intensity_image):
    return np.mean(intensity_image[regionmask])

# Pass as extra_properties=(compactness, mean_brightness) in predict_batch()
# Both region_properties and extra_properties can be combined freely.